# VideoDB Search V2 Guide

<a href="https://colab.research.google.com/github/video-db/videodb-cookbook/blob/preview/guides/indexing-v2/search/search_guide.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Build a searchable video, then move from precise retrieval to intelligent search and playable evidence.

In this guide you will:

- create transcript and structured visual indexes
- use `semantic_search()`, `query()`, and `aggregate()` directly
- let Search, DeepSearch, and Ask plan retrieval from natural language
- target an entire index or one semantic field
- inspect, play, compile, and embed timestamped results

## 1. Install dependencies

This guide uses the published VideoDB SDK.

In [ ]:
!pip install -q videodb python-dotenv pandas

## 2. Connect to VideoDB

Set `VIDEO_DB_API_KEY` in Colab secrets or your environment. If it is missing, the cell asks for it securely.

In [ ]:
import os
from getpass import getpass

import pandas as pd
from IPython.display import display
from dotenv import load_dotenv
from videodb import connect, play_stream

load_dotenv()

if not os.getenv("VIDEO_DB_API_KEY"):
    os.environ["VIDEO_DB_API_KEY"] = getpass("Enter your VideoDB API key: ")

conn = connect(api_key=os.environ["VIDEO_DB_API_KEY"])
collection = conn.get_collection()

print("Collection:", collection.id)

## 3. Choose a video

The default is a short sample clip. To use an existing video, comment out the upload and provide its ID instead.

In [ ]:
VIDEO_URL = "https://www.youtube.com/watch?v=vVlEVRKv4is"  # Silicon Valley - Gilfoyle is free for hire

video = collection.upload(url=VIDEO_URL)

# To use an existing video instead:
# VIDEO_ID = "m-..."
# video = collection.get_video(VIDEO_ID)

print("Video:", video.id)
video.play()

## 4. Create searchable video understanding

Search works over indexes. First, create two reusable analyzer outputs:

- `transcript` for spoken words
- `scene` for structured visual understanding

The VLM uses frames as its source of visual evidence and receives the transcript as supporting context. Its nested schema gives us stable fields for semantic search, filtering, and aggregation.

In [ ]:
scene_schema = {
    "scene_description": "string",
    "activity": {
        "type": "enum",
        "values": ["conversation", "using_device", "walking", "object_interaction", "other"],
    },
    "setting": {
        "type": "object",
        "fields": {
            "location_type": {
                "type": "enum",
                "values": ["office", "home", "outdoor", "vehicle", "other"],
            },
            "environment": {
                "type": "enum",
                "values": ["indoor", "outdoor", "unknown"],
            },
        },
    },
    "object_descriptions": {
        "type": "array",
        "min_items": 0,
        "max_items": 4,
        "items": {
            "name": "string",
            "description": "string",
        },
    },
}

understanding = video.understand(
    analyzers=[
        {
            "type": "spoken_words",
            "name": "transcript",
        },
        {
            "type": "vlm",
            "name": "scene",
            "inputs": ["transcript"],
            "sampling": {"strategy": "uniform", "frame_count": 6},
            "config": {
                "model": "pro",
                "prompt": (
                    "Analyze the sampled frames using the frames as the primary source of visual evidence. "
                    "Write scene_description as one clear, specific sentence describing what happens. "
                    "Classify the main activity and setting. For each notable visible object, return a short "
                    "name and a concise visual description of its appearance or role in the scene. "
                    "Use the transcript only to disambiguate the activity. Do not infer visual details from "
                    "the transcript, and do not include objects that are not visible.\n\n"
                ),
                "schema": scene_schema,
            },
        },
    ],
    segmentation={"type": "shot", "threshold": 30},
)

print("Understanding:", understanding.id)
print("Status:", understanding.status)

## 5. Wait for analyzers

Understanding runs asynchronously. Wait before indexing its outputs.

In [ ]:
understanding.wait_until_complete()

for analyzer in understanding.list_analyzers():
    print(analyzer.name, analyzer.type, analyzer.status)

transcript_analyzer = understanding.get_analyzer("transcript")
scene_analyzer = understanding.get_analyzer("scene")

## 6. Create retrieval-ready indexes

The transcript index uses derived defaults. The scene index explicitly declares which fields support semantic search, filtering, aggregation, and sorting.

Unique names make the notebook safe to rerun in the same collection.

In [ ]:
from datetime import datetime, timezone

run_suffix = datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S")
transcript_index_name = f"transcript_{run_suffix}"
scene_index_name = f"scene_{run_suffix}"

transcript_index = video.index(
    name=transcript_index_name,
    source=transcript_analyzer,
)

scene_index = video.index(
    name=scene_index_name,
    source=scene_analyzer,
    use_for=["semantic", "query", "aggregate"],
    fields={
        "semantic": [
            "scene_description",
            "activity",
            "setting.location_type",
            "object_descriptions.description",
        ],
        "filter": [
            "activity",
            "setting.location_type",
            "setting.environment",
            "object_descriptions.name",
        ],
        "aggregate": [
            "activity",
            "setting.location_type",
            "setting.environment",
            "object_descriptions.name",
        ]
    },
)

display(pd.DataFrame([
    {"Index": "Transcript", "Name": transcript_index.name, "ID": transcript_index.index_id},
    {"Index": "Scene", "Name": scene_index.name, "ID": scene_index.index_id},
]))

## 7. Wait for semantic indexes

Structured query and aggregation can become available before embeddings finish. This guide waits for `ready` so every example, including semantic search, can run.

In [ ]:
transcript_index.wait_until_complete()
scene_index.wait_until_complete()

## 8. Inspect the index contract

`fields` shows how each field can be used. `field_schema` adds the type and supported operators for each field.

In [ ]:
print("Capabilities")
display(pd.DataFrame([
    {"Capability": capability, "Fields": ", ".join(fields)}
    for capability, fields in scene_index.fields.items()
]))

print("Field schema")
display(pd.DataFrame([
    {
        "Field": field,
        "Type": schema.type,
        "Groups": ", ".join(schema.groups),
        "Operators": ", ".join(schema.operators) or "—",
    }
    for field, schema in scene_index.field_schema.items()
]))

## 9. Choose a search method

| Goal | Method | Mode | Index selection |
|---|---|---|---|
| Let VideoDB interpret the request | `search()` | `mode="default"` | VideoDB chooses |
| Investigate over multiple steps |  | `mode="deepsearch"` | VideoDB chooses |
| Search meaning directly | `semantic_search()` | — | Zero, one, or many semantic indexes |
| Apply exact conditions | `query()` | — | Exactly one index |
| Count or group values | `aggregate()` | — | Exactly one index |
| Generate an answer | `ask()` | — | VideoDB chooses |

In [ ]:
pd.options.display.max_colwidth = 120


def shots_table(shots):
    rows = [
        {
            "Video": shot.video_id,
            "Start (s)": round(shot.start, 2),
            "End (s)": round(shot.end, 2),
            "Score": round(shot.search_score, 3) if shot.search_score is not None else None,
            "Text": shot.text,
        }
        for shot in shots
    ]
    return pd.DataFrame(rows, columns=["Video", "Start (s)", "End (s)", "Score", "Text"])

## 10. Search from natural language

Use `search()` when you know the goal and want VideoDB to choose the retrieval strategy.

In [ ]:
search_response = video.search(
    query="Find the moment when an electronic device is thrown away",
    top_k=5,
    return_fields=[scene_index_name, transcript_index_name],
)

print("Response type:", search_response.response_type)
display(shots_table(search_response))

### Play one moment

`play()` opens the matching moment. `generate_stream()` returns its playable HLS URL.

In [ ]:
search_response[0].play()

### Compile and play several moments

Compile the matching shots into one stream, then pass the returned HLS URL to `play_stream()`.

In [ ]:
compiled_stream_url = search_response.results.compile()
play_stream(compiled_stream_url)

## 11. Search a semantic index

Passing an index name searches all semantic fields inside that index.

In [ ]:
semantic_results = video.semantic_search(
    query="someone handling an electronic device",
    index_names=[scene_index_name],
    top_k=5,
    score_threshold=0.2,
    return_fields=[scene_index_name],
)

display(shots_table(semantic_results))

### Target one semantic field

Append a field path to search only that field. This request searches `setting.location_type` instead of all semantic fields in the scene index.

In [ ]:
setting_results = video.semantic_search(
    query="inside a home",
    index_names=[f"{scene_index_name}.setting.location_type"],
    top_k=5,
)

display(shots_table(setting_results))

### Combine meaning with a filter

Semantic search can also require an exact indexed condition.

In [ ]:
filtered_semantic_results = video.semantic_search(
    query="people talking",
    index_names=[scene_index_name],
    filter=[
        {"field": "setting.environment", "op": "==", "value": "indoor"},
    ],
    top_k=5,
)

display(shots_table(filtered_semantic_results))

### Search one nested object field

Target the descriptions inside `object_descriptions` when the search should focus on how visible objects look or function.

In [ ]:
object_results = video.semantic_search(
    query="a small electronic device connected by a cable",
    index_names=[f"{scene_index_name}.object_descriptions.description"],
    top_k=5,
    return_fields=[scene_index_name]
)

display(shots_table(object_results))

## 12. Query exact conditions

Use `query()` when you know the index and field values. A list of conditions is an implicit AND.

In [ ]:
conversation_results = video.query(
    index_name=scene_index_name,
    filter=[
        {"field": "activity", "op": "==", "value": "conversation"},
        {"field": "setting.location_type", "op": "in", "value": ["home", "office"]},
    ],
    limit=20,
    return_fields=[scene_index_name],
)

conversation_table = pd.DataFrame([
    {
        "Start (s)": round(shot.start, 2),
        "End (s)": round(shot.end, 2),
        "Activity": (shot.metadata or {}).get("activity"),
        "Location": (shot.metadata or {}).get("setting", {}).get("location_type"),
        "Environment": (shot.metadata or {}).get("setting", {}).get("environment"),
        "Description": (shot.metadata or {}).get("scene_description"),
    }
    for shot in conversation_results
])
display(conversation_table)

### OR and NOT filters

Use explicit boolean groups for alternatives and exclusions.

In [ ]:
device_or_object_results = video.query(
    index_name=scene_index_name,
    filter={
        "and": [
            {
                "or": [
                    {"field": "activity", "op": "==", "value": "using_device"},
                    {"field": "activity", "op": "==", "value": "object_interaction"},
                ]
            },
            {
                "not": {"field": "setting.environment", "op": "==", "value": "outdoor"}
            },
        ]
    },
    limit=20,
)

display(shots_table(device_or_object_results))

## 13. Aggregate indexed fields

Aggregation returns rows instead of `Shot` objects. `group_by` is required.

In [ ]:
activity_counts = video.aggregate(
    index_name=scene_index_name,
    group_by="activity",
    metric="count",
    limit=20,
)

setting_counts = video.aggregate(
    index_name=scene_index_name,
    group_by="setting.location_type",
    metric="count",
    limit=20,
)

aggregate_table = pd.concat(
    [
        pd.DataFrame(activity_counts)
        .rename(columns={"activity": "Value", "value": "Count"})
        .assign(Breakdown="Activity"),
        pd.DataFrame(setting_counts)
        .rename(columns={"setting.location_type": "Value", "value": "Count"})
        .assign(Breakdown="Location"),
    ],
    ignore_index=True,
)[["Breakdown", "Value", "Count"]]

display(aggregate_table.style.hide(axis="index").format({"Count": "{:.0f}"}))

## 14. Investigate with DeepSearch

DeepSearch can perform multiple retrieval steps and continue through a session.

In [ ]:
deep_response = video.search(
    query="someone handling an electronic device",
    mode="deepsearch",
    top_k=10,
    return_fields="all",
)

display(pd.DataFrame([{
    "Session": deep_response.session_id,
    "Waiting for": deep_response.waiting_for,
    "Clarification": deep_response.clarification or "None",
}]))
display(shots_table(deep_response.shots))

### Continue or answer a clarification

If DeepSearch asks for more detail, answer by sending another request with the same `session_id`. You can use the same pattern for any follow-up.

In [ ]:
if deep_response.clarification:
    print("DeepSearch asks:", deep_response.clarification)

followup_response = video.search(
    query="Focus on the scenes where the device is discarded",
    mode="deepsearch",
    session_id=deep_response.session_id,
    top_k=10,
)

display(shots_table(followup_response.shots))

## 15. Ask with supporting sources

Ask returns a synthesized answer. `include_sources=True` adds the timestamped moments used as evidence.

In [ ]:
answer = video.ask(
    question="What happens to the electronic device",
    top_k=15,
    include_sources=True,
)

print("Answer:", answer.answer)
print("Sources:", len(answer.sources))
display(shots_table(answer.sources))

## 16. Inspect and play results

A `Shot` connects a match to the source video and exact time range.

In [ ]:
shots = search_response.shots

if shots:
    shot = shots[0]
    display(shots_table([shot]))
else:
    print("No shots returned")

### Play one moment

`play()` opens the matching moment. `generate_stream()` returns its playable HLS URL.

In [ ]:
shot.play()

In [ ]:
stream_url = shot.generate_stream()
play_stream(stream_url)

### Compile several moments

Direct semantic search and query return a `SearchResult`, which can compile its shots into one stream.

In [ ]:
compiled_stream_url = semantic_results.compile()
play_stream(compiled_stream_url)

For high-level Search or DeepSearch, compile the nested `SearchResult` and pass the returned HLS URL to `play_stream()`.

In [ ]:
compiled_stream_url = search_response.results.compile()
play_stream(compiled_stream_url)

## 17. Search across a collection

Replace `video.*` with `collection.*` when a result may come from any indexed video in the collection. Collection search covers the whole collection.

In [ ]:
collection_results = collection.search(
    query="Find scenes where someone handles an electronic device",
    top_k=10,
)

display(shots_table(collection_results))

## 18. Search legacy indexes

Use `legacy_search()` when migrating an existing application that still has spoken-word or scene indexes created with the legacy indexing APIs. Search V2 indexes created earlier in this guide are not used by these calls, and legacy parameters cannot be mixed with Search V2 parameters.

The examples below require a video that already has legacy indexes. Replace the placeholder ID with that video's ID. New applications should use the Search V2 methods above.

In [ ]:
from videodb import IndexType, SearchType

legacy_video = collection.get_video("m-...")

legacy_transcript_results = legacy_video.legacy_search(
    query="electronic device",
    search_type=SearchType.semantic,
    index_type=IndexType.spoken_word,
)

display(shots_table(legacy_transcript_results))

### Search a specific legacy scene index

Pass the legacy scene index ID with `scene_index_id`. `index_id` is also accepted as an alias.

In [ ]:
legacy_scene_results = legacy_video.legacy_search(
    query="someone handling an electronic device",
    search_type=SearchType.semantic,
    index_type=IndexType.scene,
    scene_index_id="i-scn-...",
)

display(shots_table(legacy_scene_results))

### Search legacy indexes across a collection

Use the same explicit legacy API at collection scope. Migration warnings returned by the server are available on `legacy_collection_results.warnings`.

In [ ]:
legacy_collection_results = collection.legacy_search(
    query="electronic device",
    search_type=SearchType.semantic,
    index_type=IndexType.spoken_word,
)

display(shots_table(legacy_collection_results))

if legacy_collection_results.warnings:
    print("Migration warnings")
    display(pd.DataFrame(legacy_collection_results.warnings))

## Quick reference

```python
# Intelligent retrieval
video.search(query="...")
video.search(query="...", mode="deepsearch")
video.ask(question="...", include_sources=True)

# Direct retrieval
video.semantic_search(query="...", index_names=["scene"])
video.query(index_name="scene", filter=[...])
video.aggregate(index_name="scene", group_by="activity")

# Legacy index compatibility
video.legacy_search(query="...", index_type=IndexType.spoken_word)
video.legacy_search(query="...", index_type=IndexType.scene, scene_index_id="i-scn-...")
collection.legacy_search(query="...", index_type=IndexType.spoken_word)

# Result actions
shot.play()
shot.generate_stream()
search_result.compile()
search_result.play()
search_result.get_embed_code()
```

### Index selection

| Method | Selector |
|---|---|
| `semantic_search()` | `index_names` / `index_ids`, zero, one, or many |
| `query()` | `index_name` or `index_id`, exactly one |
| `aggregate()` | `index_name` or `index_id`, exactly one |
| Search, DeepSearch, Ask | No index selectors |